In [2]:
import torch
from torch import nn, optim

def forsaken(f_theta_0, T, lambda_, omega, P, D_f, eta_mu, xi):
    model_t = f_theta_0
    theta_0 = [param.clone().detach() for param in f_theta_0.parameters()]
    mu = [torch.zeros_like(param, requires_grad=True) for param in theta_0]
    criterion = nn.KLDivLoss()
    optimizer = optim.LBFGS(mu, lr=eta_mu)

    for _ in range(T):
        with torch.no_grad():
            for param, theta, m in zip(model_t.parameters(), theta_0, mu):
                param.copy_((theta - xi * m).detach())

        upsilon = model_t(D_f)
        loss = criterion(upsilon, P) + lambda_ * omega * sum(torch.norm(m, p=1) for m in mu)

        loss.backward()
        optimizer.step()

    return model_t



In [17]:
###################################
# 1) Imports
###################################
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader


In [18]:

###################################
# 2) Préparation des données MNIST
###################################
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_dataset = datasets.MNIST(root="./data", train=True, transform=transform, download=True)
test_dataset = datasets.MNIST(root="./data", train=False, transform=transform, download=True)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=1000, shuffle=False)


In [20]:

###################################
# 3) Définition d'un modèle FC simple
###################################
class FullyConnectedNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(784, 256)
        self.fc2 = nn.Linear(256, 128)
        self.fc3 = nn.Linear(128, 10)

    def forward(self, x):
        x = self.flatten(x)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

model = FullyConnectedNN()

In [21]:

###################################
# 4) Entraînement rapide (optionnel)
###################################
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.01)

epochs = 2  # juste pour illustrer
for epoch in range(epochs):
    model.train()
    total_loss = 0
    for images, labels in train_loader:
        optimizer.zero_grad()
        output = model(images)
        loss = criterion(output, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss/len(train_loader):.4f}")


Epoch 1/2, Loss: 0.8119
Epoch 2/2, Loss: 0.3063


In [22]:


###################################
# 5) Fonction 'forsaken'
###################################
def forsaken(f_theta_0, T, lambda_, omega, P, D_f, eta_mu, xi):
    """
    f_theta_0 : Modèle de départ (nn.Module)
    T         : Nombre d'itérations externes
    lambda_   : Poids pour la pénalisation
    omega     : Autre hyperparam pour la pénalisation
    P         : Distribution cible (probabilités) de taille [batch_size, nb_classes]
    D_f       : Batch d'images (tensor) de taille [batch_size, C, H, W]
    eta_mu    : Taux d'apprentissage (lr) pour SGD
    xi        : Coefficient pour la mise à jour param = theta_0 - xi * mu
    """
    
    # On travaille directement sur f_theta_0 (référence)
    model_t = f_theta_0
    
    # On détache et clone les paramètres initiaux θ₀
    theta_0 = [param.clone().detach() for param in model_t.parameters()]
    
    # Vecteur mu initialisé à zéro (même dimension que θ₀)
    # On veut que mu soit "learnable", donc requires_grad=True
    mu = [torch.zeros_like(param, requires_grad=True) for param in theta_0]
    
    # Critère KLDivLoss (on utilisera log_softmax côté modèle)
    criterion_kl = nn.KLDivLoss(reduction='batchmean')
    
    # Optimiseur : SGD sur mu (pas sur les paramètres du modèle)
    optimizer_mu = optim.SGD(mu, lr=eta_mu)
    
    for _ in range(T):
        # Remettre à zéro le gradient de mu
        optimizer_mu.zero_grad()
        
        # Mettre à jour les paramètres du modèle en fonction de mu
        with torch.no_grad():
            for param, theta, m in zip(model_t.parameters(), theta_0, mu):
                param.copy_(theta - xi * m)
        
        # Forward pass : log_softmax pour KLDivLoss
        upsilon = F.log_softmax(model_t(D_f), dim=1)
        # print("upsilon", upsilon)
        # Calcul de la loss = KL(upsilon, P) + pénalisation L1 sur mu
        loss = criterion_kl(upsilon, P) + lambda_ * omega * sum(torch.norm(m, p=1) for m in mu)
        
        # Backward sur mu
        loss.backward()
        
        # Descente SGD sur mu
        optimizer_mu.step()
    
    return model_t



In [23]:
###################################
# 6) Exemple d'utilisation
###################################
# On prend un batch de 20 images du train_loader
images, labels = next(iter(train_loader))
images = images[:20]
labels = labels[:20]

# Distribution uniforme : [0.1, ..., 0.1] pour chaque image
# (taille = [batch_size, nb_classes])
P = torch.full((images.size(0), 10), 1/10)

# Paramètres Forsaken
T = 5         # Nombre d'itérations externes
lambda_ = 0.1
omega = 1.0
eta_mu = 0.01
xi = 0.05

# Lancement de Forsaken
model_forsaken = forsaken(
    f_theta_0=model,
    T=T,
    lambda_=lambda_,
    omega=omega,
    P=P,
    D_f=images,
    eta_mu=eta_mu,
    xi=xi
)

print("Fin de la procédure Forsaken.")

Fin de la procédure Forsaken.


In [24]:
import torch.nn.functional as F

# On évalue le modèle "oublié" sur les mêmes 20 images
with torch.no_grad():
    # Sortie brute du modèle
    outputs_f = model_forsaken(images)
    
    # Probabilités prédites
    probs_f = F.softmax(outputs_f, dim=1)
    print("Probabilités prédites sur les points oubliés :")
    print(probs_f)
    
    # Calcul de la divergence KL entre les log-probabilités du modèle et la distribution cible P
    log_probs_f = F.log_softmax(outputs_f, dim=1)
    criterion_kl = nn.KLDivLoss(reduction='batchmean')
    kl_div = criterion_kl(log_probs_f, P)
    print(f"KL divergence entre la sortie du modèle et la distribution cible : {kl_div.item():.4f}")
    
    # Optionnel : Calcul de l'accuracy (les prédictions uniformes devraient rendre l'accuracy proche du hasard, ~10% pour MNIST)
    _, preds_f = torch.max(outputs_f, dim=1)
    accuracy_f = (preds_f == labels).sum().item() / len(labels) * 100
    print(f"Accuracy sur les points oubliés : {accuracy_f:.2f}%")


Probabilités prédites sur les points oubliés :
tensor([[5.0260e-04, 2.4849e-06, 8.4838e-04, 2.3001e-02, 1.4401e-04, 8.9527e-01,
         1.6426e-05, 5.4640e-06, 7.5691e-02, 4.5143e-03],
        [1.2252e-04, 1.7597e-04, 5.4235e-02, 9.3800e-01, 2.0158e-05, 3.3360e-03,
         6.7007e-05, 2.7479e-06, 4.0106e-03, 2.8332e-05],
        [2.8258e-02, 4.4859e-04, 3.5606e-02, 8.3020e-03, 7.2427e-01, 4.2031e-02,
         4.3972e-02, 4.7530e-04, 8.1107e-02, 3.5526e-02],
        [2.5820e-02, 2.4974e-04, 4.1210e-03, 4.5768e-01, 5.1084e-05, 4.9704e-01,
         1.6639e-04, 1.1705e-04, 1.4720e-02, 3.3218e-05],
        [3.5091e-01, 3.0428e-03, 1.1987e-01, 7.7752e-02, 2.5114e-05, 2.0976e-01,
         1.3410e-03, 2.4930e-03, 2.3271e-01, 2.0912e-03],
        [1.5822e-05, 2.2252e-06, 1.4520e-04, 9.8240e-01, 9.3913e-06, 1.7131e-03,
         1.7483e-08, 4.0701e-05, 9.8207e-03, 5.8563e-03],
        [9.1805e-01, 7.6592e-07, 2.6410e-04, 2.6731e-04, 5.2568e-07, 8.0902e-02,
         1.2668e-05, 5.4920e-06, 4.857